# 1. What you'll learn

        - implement QKᵀ scaling and softmax by hand
- visualize a causal attention matrix
- train a tiny character model without training any LLM

        **The one question this notebook answers:** How does scaled dot-product attention route information, and why must language training hide future characters?

# 2. Setup

This lesson keeps attention and a tiny character transformer small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A short bundled public-domain-style nursery verse is repeated only to form character windows. This is a tiny mechanism demo, not an LLM; context length is 24 and the model is under 40k parameters.

**Small example:** For five key scores [2,1,0,-1,-2], softmax gives roughly [0.64,.23,.09,.03,.01]. Attention returns that weighted average of the five value vectors.

In [ ]:
text=("twinkle twinkle little star how i wonder what you are up above the world so high like a diamond in the sky. ")*35;chars=sorted(set(text));stoi={c:i for i,c in enumerate(chars)};itos=dict(enumerate(chars));ids=torch.tensor([stoi[c] for c in text],dtype=torch.long);cut=int(.85*len(ids));print("Characters:",len(text),"vocabulary:",len(chars),chars,"train/validation:",cut,len(ids)-cut)

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
counts=np.bincount(ids.numpy(),minlength=len(chars));pairs=np.zeros((len(chars),len(chars)),int)
for a,b in zip(ids[:-1],ids[1:]): pairs[a,b]+=1
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].bar(range(len(chars)),counts,color="#176b66");axes[0].set_xticks(range(len(chars)),chars);axes[0].set_title("Character frequency")
axes[1].imshow(pairs,cmap="mako");axes[1].set_title("Next-character counts");axes[2].plot(ids[:160]);axes[2].set_title("Character IDs are sequence, not magnitude");plt.tight_layout();plt.show()

**Takeaway:** Spaces and common vowels make a strong unigram baseline.

**Takeaway:** Adjacent-character structure already narrows plausible next symbols.

**Takeaway:** Integer IDs are lookup keys; their numeric distance carries no meaning.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
torch.manual_seed(SEED);context=24;vocab=len(chars)
def batch(split,batch_size=32):
 data=ids[:cut] if split=="train" else ids[cut-context:];starts=torch.randint(0,len(data)-context-1,(batch_size,));x=torch.stack([data[s:s+context] for s in starts]);y=torch.stack([data[s+1:s+context+1] for s in starts]);return x,y
x,y=batch("train");print("Batch inputs/targets:",x.shape,y.shape,"first pair:","".join(itos[int(i)] for i in x[0][:10]),"→","".join(itos[int(i)] for i in y[0][:10]))

# 6. Train

        Training turns the assumptions behind attention and a tiny character transformer into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Embed character identity and position.
2. Project hidden vectors into queries, keys, and values.
3. Mask future positions, scale QKᵀ, and softmax into attention weights.
4. Combine attended values, pass through an MLP, and predict every next character.

In [ ]:
d=24
class TinyTransformer(nn.Module):
 def __init__(self,causal=True):
  super().__init__();self.causal=causal;self.tok=nn.Embedding(vocab,d);self.pos=nn.Embedding(context,d);layer=nn.TransformerEncoderLayer(d,4,48,batch_first=True,dropout=0);self.block=nn.TransformerEncoder(layer,1);self.head=nn.Linear(d,vocab)
 def forward(self,x):
  z=self.tok(x)+self.pos(torch.arange(x.shape[1]));mask=nn.Transformer.generate_square_subsequent_mask(x.shape[1]) if self.causal else None;return self.head(self.block(z,mask=mask))
model=TinyTransformer(True);opt=torch.optim.AdamW(model.parameters(),lr=.004);losses=[]
for step in range(220):
 x,y=batch("train");opt.zero_grad();logits=model(x);loss=nn.functional.cross_entropy(logits.reshape(-1,vocab),y.reshape(-1));loss.backward();opt.step();losses.append(loss.item())
print("Parameters:",sum(p.numel() for p in model.parameters()),"loss first/last:",round(losses[0],3),round(losses[-1],3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
model.eval();vx,vy=batch("validation",64)
with torch.no_grad(): vloss=nn.functional.cross_entropy(model(vx).reshape(-1,vocab),vy.reshape(-1)).item()
unigram=-np.sum((counts/counts.sum())*np.log(counts/counts.sum()+1e-12));Q=torch.tensor([[1.,0.],[.7,.3],[0.,1.],[-.5,.5],[.2,.8]]);scores=Q@Q.T/np.sqrt(2);mask=torch.triu(torch.ones(5,5)*float("-inf"),diagonal=1);weights=torch.softmax(scores+mask,dim=1)
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(losses);axes[0].axhline(unigram,ls="--",color="black",label="unigram entropy");axes[0].set_title("Next-character loss");axes[0].legend()
im=axes[1].imshow(weights,cmap="mako",vmin=0,vmax=1);fig.colorbar(im,ax=axes[1]);axes[1].set(title="Hand-made causal attention",xlabel="key position",ylabel="query position")
axes[2].bar(["unigram perplexity","transformer"],[np.exp(unigram),np.exp(vloss)],color=[".6","#176b66"]);axes[2].set_title("Validation perplexity");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Validation perplexity={np.exp(vloss):.2f}; unigram={np.exp(unigram):.2f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Without a causal mask, next-token training leaks future characters and reports an invalidly low loss. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
leaky=TinyTransformer(False);leaky_opt=torch.optim.AdamW(leaky.parameters(),lr=.004);leaky_losses=[]
for step in range(100):
 x,y=batch("train");leaky_opt.zero_grad();loss=nn.functional.cross_entropy(leaky(x).reshape(-1,vocab),y.reshape(-1));loss.backward();leaky_opt.step();leaky_losses.append(loss.item())
with torch.no_grad(): leaked_val=nn.functional.cross_entropy(leaky(vx).reshape(-1,vocab),vy.reshape(-1)).item()
print(f"Causal validation loss={vloss:.3f}; future-visible loss={leaked_val:.3f}. The latter cheats by attending to characters whose shifted targets reveal the answer.")

**Use this when…** sequence elements need content-dependent long-range interaction and parallel training is valuable.

        **Don't use this when…** data/compute are tiny, a lag baseline wins, or the task needs true online constant-memory state.

        ## Try this

        1. Remove √d scaling and inspect softmax saturation.
2. Change context from 8 to 48.
3. Generate characters autoregressively at temperatures .5 and 1.2.